# Sequence Neural Net (1D-CNN / LSTM) — Proposal Draft for `Fault_Within_6h`

**Status: executed locally, results below.** Trained on the real dataset (Kaggle pull + `tensorflow`/`numpy` installed) — see Section 7 for the actual macro F1 / recall numbers, and Section 8 for how they compare to the Random Forest/XGBoost baseline in `03_sequence_classifier.ipynb`.

## Why this notebook exists

`01b_sequence_features.ipynb` and `03_sequence_classifier.ipynb` reframed the task from "is a fault happening right now" to "will a fault happen in the next 6 hours" (`Fault_Within_6h`), and fed **hand-engineered** rolling/lag features (6h/12h/24h rolling mean/std, 1h/3h/6h lags, 6h delta) into Random Forest / XGBoost.

The dataset's own README (`data/raw/driving_pattern_diagnostics/README.md`) states its intended use case is *"multivariate time-series forecasting (e.g., CNN + LSTM models)"*, and the project plan (`docs/Capstone_Project_Plan.md`, Section 3) already lists the stack as "baseline → small neural net (MLP/1D-CNN)". This notebook is that next step, not a replacement of the baseline.

**What changes:** instead of hand-computed rolling statistics, the model is given the **raw sequence** of the last `WINDOW` hours directly (a small table of hours × sensors per example) and learns its own temporal patterns from it — a 1D-CNN scans for local shapes (e.g., a sharp short spike), an LSTM tracks longer-running trends (e.g., a slow multi-day drift).

**What stays the same:** same dataset, same `Fault_Within_6h` target, same chronological per-vehicle split discipline, same class-weighting-from-train-fold-only discipline, same evaluation metrics (macro F1, per-class recall, confusion matrix) — so results are directly comparable to the Random Forest/XGBoost numbers in `03_sequence_classifier.ipynb`.

In [16]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
keras.utils.set_random_seed(42)

## 1. Config

Reusing the same 4 sensors `01b_sequence_features.ipynb` already flagged as most predictive for the Fault/Warning tiers (`Motor_RPM`, `Motor_Torque`, `Motor_Temp`, `Battery_Temp`), rather than all 10 columns — keeps the first pass small, consistent with the "small neural net" framing in the plan. `WINDOW` (hours of trailing history per example) is a tunable knob — 24h is a reasonable starting point since it matches the longest rolling window already used in the baseline features.

In [17]:
SEQ_SENSORS = ["Motor_RPM", "Motor_Torque", "Motor_Temp", "Battery_Temp"]
WINDOW = 24  # trailing hours of history per example
TARGET = "Fault_Within_6h"

## 2. Load data, chronological per-vehicle split

Same split function as `01b_sequence_features.ipynb` / `03_sequence_classifier.ipynb` — last ~20% of each vehicle's own timeline as test, so the model is always evaluated on the future relative to what it trained on.

**Important difference from the baseline notebooks:** the split happens *before* windowing here, not after. Each vehicle's train/test hours are separated first, and sliding windows are built independently within each half. This means the first `WINDOW` hours of each vehicle's test period can't form a full window (no test-only history yet) and are dropped — a small, deliberate loss of a few rows per vehicle, not an oversight, and the safest way to guarantee no train-side hours leak into a test window.

In [18]:
df = pd.read_csv(
    "../data/processed/driving_pattern_diagnostics_sequence_features.csv",
    parse_dates=["timestamp"],
)
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(bool)
df = df.sort_values(["user_profile", "timestamp"]).reset_index(drop=True)


def chronological_split(data, group_col="user_profile", time_col="timestamp", test_frac=0.2):
    train_parts, test_parts = [], []
    for _, g in data.groupby(group_col):
        g = g.sort_values(time_col)
        cutoff = int(len(g) * (1 - test_frac))
        train_parts.append(g.iloc[:cutoff])
        test_parts.append(g.iloc[cutoff:])
    return pd.concat(train_parts).reset_index(drop=True), pd.concat(test_parts).reset_index(drop=True)


train_df, test_df = chronological_split(df)
print(f"Train: {len(train_df):,} rows, Test: {len(test_df):,} rows")

Train: 140,064 rows, Test: 35,020 rows


## 3. Build sliding windows

For each vehicle, slide a `WINDOW`-hour block across its own timeline only — grouped by `user_profile` so a window can never mix hours from two different vehicles, same discipline as the rolling features in `01b_sequence_features.ipynb`.

In [19]:
def make_windows(data, sensors, window, target):
    X, y = [], []
    for _, g in data.groupby("user_profile"):
        g = g.sort_values("timestamp")
        values = g[sensors].to_numpy()
        labels = g[target].to_numpy()
        for i in range(window, len(g)):
            X.append(values[i - window:i])
            y.append(labels[i])
    return np.array(X), np.array(y)


X_train, y_train = make_windows(train_df, SEQ_SENSORS, WINDOW, TARGET)
X_test, y_test = make_windows(test_df, SEQ_SENSORS, WINDOW, TARGET)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"{TARGET} rate — train: {y_train.mean()*100:.3f}%, test: {y_test.mean()*100:.3f}%")

X_train: (139968, 24, 4), X_test: (34924, 24, 4)
Fault_Within_6h rate — train: 8.977%, test: 9.386%


## 4. Scale

Fit the scaler on the train windows only, same leakage discipline as the sklearn `Pipeline`s in `02_baseline_classifier.ipynb`/`03_sequence_classifier.ipynb` — never fit on test.

In [20]:
n_train, w, n_feat = X_train.shape
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.reshape(-1, n_feat)).reshape(n_train, w, n_feat)
X_test_scaled = scaler.transform(X_test.reshape(-1, n_feat)).reshape(X_test.shape[0], w, n_feat)

## 5. Models — 1D-CNN and LSTM, small on purpose

Two small architectures for the same input/output shape, so they can be trained and compared side by side rather than committing to one blind. The 1D-CNN scans short local windows for shapes (e.g. a sharp spike); the LSTM reads the sequence step by step and carries a running summary forward (better suited to slower, longer-running trends).

In [21]:
def build_cnn_model(window, n_features):
    return keras.Sequential([
        layers.Input(shape=(window, n_features)),
        layers.Conv1D(filters=32, kernel_size=3, activation="relu"),
        layers.Conv1D(filters=32, kernel_size=3, activation="relu"),
        layers.GlobalMaxPooling1D(),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ], name="cnn_1d")


def build_lstm_model(window, n_features):
    return keras.Sequential([
        layers.Input(shape=(window, n_features)),
        layers.LSTM(32),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ], name="lstm")

## 6. Train

Class weight computed from the train fold only (same imbalance-handling discipline as `compute_sample_weight` in `03_sequence_classifier.ipynb`), plus early stopping since these are small models on a small first pass — no need to hand-pick an epoch count.

In [22]:
fault_rate = y_train.mean()
class_weight = {0: 1.0, 1: (1 - fault_rate) / fault_rate}
print("class_weight:", class_weight)

models = {"cnn_1d": build_cnn_model(WINDOW, n_feat), "lstm": build_lstm_model(WINDOW, n_feat)}
histories = {}

for name, model in models.items():
    print(f"=== training {name} ===")
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=[keras.metrics.Recall(name="recall")])
    histories[name] = model.fit(
        X_train_scaled, y_train,
        validation_split=0.15,
        epochs=20,
        batch_size=256,
        class_weight=class_weight,
        callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
        verbose=2,
    )

class_weight: {0: 1.0, 1: np.float64(10.139514524472741)}
=== training cnn_1d ===
Epoch 1/20
465/465 - 3s - 7ms/step - loss: 1.2224 - recall: 0.6038 - val_loss: 0.5687 - val_recall: 0.2125
Epoch 2/20
465/465 - 1s - 3ms/step - loss: 1.1811 - recall: 0.6430 - val_loss: 0.5639 - val_recall: 0.2608
Epoch 3/20
465/465 - 2s - 4ms/step - loss: 1.1617 - recall: 0.6642 - val_loss: 0.5789 - val_recall: 0.3213
Epoch 4/20
465/465 - 1s - 3ms/step - loss: 1.1476 - recall: 0.6780 - val_loss: 0.5781 - val_recall: 0.3444
Epoch 5/20
465/465 - 1s - 3ms/step - loss: 1.1362 - recall: 0.6874 - val_loss: 0.5747 - val_recall: 0.3530
=== training lstm ===
Epoch 1/20
465/465 - 7s - 14ms/step - loss: 1.1062 - recall: 0.7884 - val_loss: 0.5188 - val_recall: 0.7810
Epoch 2/20
465/465 - 4s - 8ms/step - loss: 1.0647 - recall: 0.8188 - val_loss: 0.5232 - val_recall: 0.7990
Epoch 3/20
465/465 - 4s - 9ms/step - loss: 1.0569 - recall: 0.8328 - val_loss: 0.5206 - val_recall: 0.8040
Epoch 4/20
465/465 - 4s - 8ms/step - lo

## 7. Evaluate

Same metrics as `03_sequence_classifier.ipynb` — macro F1, recall on the positive (fault-within-6h) class, and a confusion matrix — so these numbers drop straight into the same comparison table as the Random Forest/XGBoost results.

In [24]:
results = {}
for name, model in models.items():
    y_pred = (model.predict(X_test_scaled) > 0.5).astype(int).ravel()
    print("=" * 60)
    print(name.upper())
    print("=" * 60)
    print(classification_report(y_test, y_pred, digits=3))
    print(confusion_matrix(y_test, y_pred))
    results[name] = {
        "macro_F1": f1_score(y_test, y_pred, average="macro"),
        "recall_positive": recall_score(y_test, y_pred, pos_label=1),
    }

pd.DataFrame(results).T

1092/1092 ━━━━━━━━━━━━━━━━━━━━ 1s 898us/step
CNN_1D
              precision    recall  f1-score   support

       False      0.943     0.674     0.787     31646
        True      0.162     0.609     0.256      3278

    accuracy                          0.668     34924
   macro avg      0.553     0.642     0.521     34924
weighted avg      0.870     0.668     0.737     34924

[[21344 10302]
 [ 1283  1995]]
1092/1092 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
LSTM
              precision    recall  f1-score   support

       False      0.968     0.616     0.753     31646
        True      0.178     0.803     0.291      3278

    accuracy                          0.633     34924
   macro avg      0.573     0.709     0.522     34924
weighted avg      0.894     0.633     0.709     34924

[[19480 12166]
 [  647  2631]]


,macro_F1,recall_positive
cnn_1d,0.521361,0.608603
lstm,0.521818,0.802624


## 8. Results vs. the baseline, and open questions for the team

**How these compare to the Random Forest/XGBoost baseline (`03_sequence_classifier.ipynb`, ~0.60 macro F1 / ~46-59% recall on `Fault_Within_6h`):**

- **Macro F1 is lower here** — CNN-1D 0.521, LSTM 0.522, vs. ~0.60 for the baseline. On this metric the hand-engineered rolling/lag features currently win.
- **Recall on the fault class is notably higher for the LSTM** — 80.3%, vs. ~46-59% baseline — it catches far more real faults, at the cost of more false alarms (precision on the fault class is only 0.178). CNN-1D's recall (60.9%) is a smaller improvement over the baseline.
- **Read together, not in isolation:** the LSTM result is a real, explainable trade-off (heavier class weighting pushes it to flag more aggressively), not an unambiguous win or loss — worth presenting as "which metric matters more for a fleet manager" rather than a single winner/loser verdict.

**Open questions for the team:**

- Is `WINDOW = 24` the right length, or should we sweep a few values (e.g. 12h/48h)?
- Worth trying a combined CNN→LSTM model (convolution first to extract local shapes, LSTM on top to track how those shapes evolve) given neither model alone clearly beats the baseline on macro F1.
- Given the LSTM's much higher recall, is it worth tuning the decision threshold (currently the default 0.5) on either model to find a better recall/precision balance before declaring a winner?